<a href="https://colab.research.google.com/github/AnwX73/MOHID-Arabic-Dialect-Normalization/blob/main/code/MOHID_Interface_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:#8B1E2D;padding:26px;border-radius:18px;color:white;text-align:center'>
<h1 style='margin:0;color:white;'>مُوحِّد | MOHID</h1>
<h3 style='margin:8px 0 0 0;color:#FCE8EB;'>Arabic Dialect → Modern Standard Arabic</h3>
</div>

<div style='border-left:6px solid #8B1E2D;padding:12px 16px;margin-top:18px;background:#FFF7F8;border-radius:10px'>
<b>Demo Notebook</b><br>
This notebook loads the already trained MOHID model and converts Arabic dialect sentences into Modern Standard Arabic (MSA).<br>
<b>No training is performed here.</b>
</div>


## 1. Setup
Install only the libraries needed to run the trained model.

In [ ]:
# Install the libraries needed to run MOHID

!pip install -q "transformers==4.46.3" "sentencepiece>=0.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 43.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Load the Final MOHID Model Files

The trained MOHID model is downloaded from the shared project model file.

In [ ]:
# Download and extract the final MOHID model

!pip install -q gdown

import os
import zipfile
import gdown

file_id = "1yuxlgq3NNuNYgg36MJcz1hsCyRhyeNRr"

zip_path = "/content/MOHID_Final_Model.zip"
extract_dir = "/content/MOHID_Final_Model"

gdown.download(
    id=file_id,
    output=zip_path,
    quiet=False
)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

model_path = "/content/MOHID_Final_Model/MOHID_Final_Model"

print("MOHID model files are ready!")
print("Model path:", model_path)

Downloading...
From (original): https://drive.google.com/uc?id=1yuxlgq3NNuNYgg36MJcz1hsCyRhyeNRr
From (redirected): https://drive.google.com/uc?id=1yuxlgq3NNuNYgg36MJcz1hsCyRhyeNRr&confirm=t&uuid=16679a07-45d4-4d66-90ba-b0f2036f7f0b
To: /content/MOHID_Final_Model.zip
100%|██████████| 1.23G/1.23G [00:11<00:00, 105MB/s] 


MOHID model files are ready!
Model path: /content/MOHID_Final_Model/MOHID_Final_Model


## 3. Load the Final Model
The notebook automatically uses GPU when available and CPU otherwise.

In [ ]:
# Load the trained MOHID model and tokenizer

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model.to(device)
model.eval()

print("MOHID loaded successfully!")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MOHID loaded successfully!
Device: cpu


## 4. MOHID Normalization Function
The same task instruction used during fine-tuning is kept here.

In [ ]:
# Create the dialect-to-MSA normalization function

task_prefix = "حوّل اللهجة العربية إلى العربية الفصحى مع الحفاظ على المعنى: "

def normalize_to_msa(text):
    text = str(text).strip()
    if not text:
        return "الرجاء إدخال جملة عربية."

    input_text = task_prefix + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

<div style='background:#FFF7F8;border:2px solid #8B1E2D;padding:18px;border-radius:16px'>
<h2 style='color:#8B1E2D;margin-top:0;'>جرّب مُوحِّد</h2>
اكتب جملة باللهجة العربية ثم اضغط <b>حوّل إلى الفصحى</b>.
</div>

In [ ]:
# Create a polished red-themed MOHID demo interface

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML("""
<div style="
    direction:rtl;
    text-align:center;
    background:linear-gradient(135deg,#6E1423,#A52A3A);
    padding:28px 20px;
    border-radius:20px 20px 0 0;
    color:white;
">
    <div style="font-size:34px;font-weight:800;">مُوحِّد</div>
    <div style="font-size:15px;margin-top:5px;opacity:.9;">MOHID</div>
    <div style="font-size:18px;margin-top:12px;">
        من اللهجة العربية إلى العربية الفصحى
    </div>
</div>
""")

input_label = widgets.HTML("""
<div style="
    direction:rtl;
    text-align:right;
    font-size:16px;
    font-weight:700;
    margin:16px 4px 8px 4px;
">
    اكتب الجملة باللهجة العربية
</div>
""")

input_box = widgets.Textarea(
    placeholder="مثال: أبي أغير موعد الحجز",
    layout=widgets.Layout(
        width="100%",
        height="110px"
    )
)

convert_button = widgets.Button(
    description="حوّل إلى الفصحى",
    icon="language",
    layout=widgets.Layout(
        width="230px",
        height="48px",
        margin="16px auto"
    )
)

convert_button.style.button_color = "#8B1E2D"
convert_button.style.font_weight = "bold"

result_box = widgets.HTML("""
<div style="
    direction:rtl;
    text-align:right;
    background:#FFF7F8;
    border:1px solid #E6C1C7;
    border-radius:14px;
    padding:18px;
    margin-top:8px;
    min-height:55px;
">
    <span style="color:#8B1E2D;font-weight:700;">النتيجة بالفصحى</span>
    <div style="margin-top:8px;font-size:18px;color:#555;">
        ستظهر النتيجة هنا...
    </div>
</div>
""")

footer = widgets.HTML("""
<div style="
    direction:rtl;
    text-align:center;
    color:#888;
    font-size:12px;
    padding:14px;
">
    MOHID • Arabic Dialect Normalization
</div>
""")

def run_demo(_):
    text = input_box.value.strip()

    if not text:
        result_box.value = """
        <div style="
            direction:rtl;
            text-align:right;
            background:#FFF7F8;
            border:1px solid #E6C1C7;
            border-radius:14px;
            padding:18px;
        ">
            <span style="color:#8B1E2D;font-weight:700;">
                الرجاء كتابة جملة أولاً
            </span>
        </div>
        """
        return

    result = normalize_to_msa(text)

    result_box.value = f"""
    <div style="
        direction:rtl;
        text-align:right;
        background:#FFF7F8;
        border:2px solid #8B1E2D;
        border-radius:14px;
        padding:20px;
    ">
        <span style="color:#8B1E2D;font-weight:700;font-size:15px;">
            الجملة بالفصحى
        </span>

        <div style="
            margin-top:10px;
            font-size:21px;
            font-weight:600;
            color:#222;
        ">
            {result}
        </div>
    </div>
    """

convert_button.on_click(run_demo)

interface = widgets.VBox(
    [
        header,
        input_label,
        input_box,
        convert_button,
        result_box,
        footer
    ],
    layout=widgets.Layout(
        width="100%",
        max_width="1100px",
        min_height="650px",
        margin="20px auto",
        padding="0 18px 10px 18px",
        border="1px solid #DDD",
    )
)

display(interface)